# Probabilistic Primality Testing: The Miller–Rabin Test

## Your goals:

By the end of this notebook, you should be able to:

1. explain why fast primality testing is needed in RSA;
2. distinguish **prime**, **composite**, and **probably prime**;
3. understand the mathematics behind one Miller–Rabin round;
4. implement the test in Python;
5. explain its correctness, probability of error, and running time.



## 1. Why is this useful in RSA?

RSA begins by choosing two large prime numbers, usually denoted by $p$ and $q$, and forming

$$n=pq.$$

For small numbers, we could test primality by trying divisors up to $\sqrt n$. For the very large candidates used in RSA, that direct method is far too slow. Therefore, an RSA key generator repeatedly does the following:

1. generate a large random odd candidate $p$;
2. quickly reject obvious composites, for example numbers divisible by small primes;
3. apply a fast primality test such as Miller–Rabin;
4. accept the candidate only after it passes enough tests;
5. repeat the process to obtain $q$.

Miller–Rabin is therefore useful during **RSA key generation**. It is not the RSA encryption or decryption algorithm itself. It helps find numbers that can safely be treated as primes with an extremely small probability of error.

## 2. Basic mathematical ideas

### 2.1 Prime and composite numbers

An integer $n>1$ is **prime** if its only positive divisors are $1$ and $n$. Otherwise it is **composite**.

Examples:

- $13$ is prime.
- $21=3\cdot 7$ is composite.

### 2.2 Congruence modulo $n$

The notation

$$x\equiv y\pmod n$$

means that $x$ and $y$ leave the same remainder when divided by $n$. Equivalently, $n$ divides $x-y$.

For example,

$$38\equiv 3\pmod 7$$

because both numbers leave remainder $3$ on division by $7$.

### 2.3 Fermat's little theorem

If $n$ is prime and $1\le a<n$, then

$$a^{n-1}\equiv 1\pmod n.$$

Thus, if we ever find an $a$ for which $a^{n-1}\not\equiv 1\pmod n$, then $n$ is certainly composite.

The converse is not always true: some composite numbers also satisfy the Fermat congruence for certain bases. Miller–Rabin performs a stronger check.

## 3. The key observation behind Miller–Rabin

Let $n>2$ be odd. Then $n-1$ is even, so we can repeatedly remove factors of $2$ and write uniquely

$$n-1=2^s d,$$

where $d$ is odd.

For example, if $n=13$, then

$$12=2^2\cdot 3,$$

so $s=2$ and $d=3$.

Now choose a base $a$ with $2\le a\le n-2$ and compute

$$x_0=a^d\bmod n.$$

Then square repeatedly:

$$x_1=x_0^2\bmod n,\quad x_2=x_1^2\bmod n,\quad\ldots$$

The exponents become

$$d,\;2d,\;2^2d,\;\ldots,\;2^sd=n-1.$$

If $n$ is prime, Fermat's theorem says the last value is $1$. More importantly, for a prime $n$, the only solutions of

$$x^2\equiv 1\pmod n$$

are $x\equiv 1$ and $x\equiv -1\equiv n-1\pmod n$.

Therefore, along the squaring chain for a prime, we must either:

- start at $1$, or
- meet $-1$ before the chain reaches $1$.

If neither happens, the chosen base exposes that $n$ is composite.

## 4. Two small illustrations

### 4.1 Prime example: $n=13$, base $a=2$

We have $13-1=12=2^2\cdot 3$, so $s=2$ and $d=3$.

$$x_0=2^3\bmod 13=8.$$

The first value is neither $1$ nor $12=-1\pmod{13}$, so square it:

$$x_1=8^2\bmod 13=64\bmod 13=12=-1\pmod{13}.$$

The round passes, as expected for the prime $13$.

### 4.2 Composite example: $n=21$, base $a=2$

We have $21-1=20=2^2\cdot 5$, so $s=2$ and $d=5$.

$$x_0=2^5\bmod 21=32\bmod 21=11,$$

and then

$$x_1=11^2\bmod 21=121\bmod 21=16.$$

Neither $1$ occurs initially nor does $20=-1\pmod{21}$ occur in the permitted chain. Hence base $2$ is a **witness** that $21$ is composite.

A base that makes a composite number pass is called a **strong liar** for that number. This possibility is why the test is probabilistic and why several independent rounds are used.

In [105]:
def write_as_power_of_two(n):
    """Find s and d such that n - 1 = 2**s * d, where d is odd."""
    d = n - 1
    s = 0

    while d % 2 == 0:
        d = d // 2
        s = s + 1

    return s, d

In [106]:
# Printing the decomposition for the earlier examples
for n in (13, 21):
    s, d = write_as_power_of_two(n)
    print(f"{n} - 1 = 2^{s} * {d}")

13 - 1 = 2^2 * 3
21 - 1 = 2^2 * 5


## 5. One Miller–Rabin round

For an odd integer $n>3$:

1. Write $n-1=2^s d$, where $d$ is odd.
2. Choose a base $a$ uniformly from $\{2,3,\ldots,n-2\}$.
3. Compute $x=a^d\bmod n$.
4. If $x=1$ or $x=n-1$, the round passes.
5. Otherwise, repeat at most $s-1$ times:
   - replace $x$ by $x^2\bmod n$;
   - if $x=n-1$, the round passes.
6. If $n-1$ was never obtained, return **composite**.

Terminology:

- If a round fails, the base is a **witness**, and $n$ is definitely composite.
- If a round passes, $n$ is a **strong probable prime to base $a$**.
- Passing does not prove primality, so after several rounds the answer is **probably prime**.

In [107]:
def miller_rabin_round(n, a):
    """Perform one Miller-Rabin round using base a."""

    # Step 1: Write n - 1 = 2**s * d, where d is odd.
    s, d = write_as_power_of_two(n)

    # Step 2: Compute x = a**d mod n.
    x = pow(a, d, n)

    # Step 3: The round passes if x is 1 or -1 modulo n.
    if x == 1 or x == n - 1:
        return True

    # Step 4: Square x at most s - 1 times.
    for _ in range(s - 1):
        x = (x * x) % n

        # The round passes if -1 modulo n is obtained.
        if x == n - 1:
            return  True

    # Step 5: No required value was obtained; n is composite.
    return False

if miller_rabin_round(2047, 2): print ("Probably prime") 
else: print ("Surely Composite") 

#if miller_rabin_round(2047, 3): print ("Probably prime") 
#else: print ("Surely Composite") 


Probably prime


In [108]:
def print_miller_rabin_round(n, a):
    """Print the calculations of one Miller-Rabin round."""
    s, d = write_as_power_of_two(n)
    x = pow(a, d, n)

    print(f"n - 1 = 2^{s} * {d}")
    print(f"x0 = {a}^{d} mod {n} = {x}")

    if x == 1:
        print("The round passes because x0 = 1.")
        return True

    if x == n - 1:
        print(f"The round passes because x0 = {n - 1} = -1 mod {n}.")
        return True

    for i in range(1, s):
        x = (x * x) % n
        print(f"x{i} = previous value squared mod {n} = {x}")

        if x == n - 1:
            print(f"The round passes because x{i} = {n - 1} = -1 mod {n}.")
            return True

    print("The round fails; therefore, the number is surely composite.")
    return False


print("Testing 13 with base 2:")
print_miller_rabin_round(13, 2)

print("\nTesting 21 with base 2:")
print_miller_rabin_round(21, 2)

print("\nTesting 21 with base 2:")
if print_miller_rabin_round(2047, 2): print ("Probably True")


Testing 13 with base 2:
n - 1 = 2^2 * 3
x0 = 2^3 mod 13 = 8
x1 = previous value squared mod 13 = 12
The round passes because x1 = 12 = -1 mod 13.

Testing 21 with base 2:
n - 1 = 2^2 * 5
x0 = 2^5 mod 21 = 11
x1 = previous value squared mod 21 = 16
The round fails; therefore, the number is surely composite.

Testing 21 with base 2:
n - 1 = 2^1 * 1023
x0 = 2^1023 mod 2047 = 1
The round passes because x0 = 1.
Probably True


### 5.1 A genuine strong-liar example: $n=2047$ and base $a=2$

The number

$$2047=23\cdot 89$$

is composite. Nevertheless, base $a=2$ makes one Miller–Rabin round pass.

First write

$$2047-1=2046=2^1\cdot 1023,$$

so $s=1$ and $d=1023$. Now compute

$$x_0=2^{1023}\bmod 2047=1.$$

Miller–Rabin accepts the round immediately because $x_0=1$. Thus, **2 is a strong liar for 2047**. It hides the compositeness of 2047 in this particular round.

This does not mean that every base lies. For example, base $3$ exposes that 2047 is composite. Randomly selecting several independent bases makes it very unlikely that all selected bases will be strong liars.

In [109]:
n = 2047
print("2047 = 23 * 89, so it is composite.\n")

for a in (2, 3):
    print(f"Testing base a = {a}")
    passed = print_miller_rabin_round(n, a)
    if passed:
        print(f"Base {a} passes: it is a strong liar for {n}." if a == 2 else f"Base {a} passes.")
    else:
        print(f"Base {a} is a witness: it proves that {n} is composite.")
    print()

2047 = 23 * 89, so it is composite.

Testing base a = 2
n - 1 = 2^1 * 1023
x0 = 2^1023 mod 2047 = 1
The round passes because x0 = 1.
Base 2 passes: it is a strong liar for 2047.

Testing base a = 3
n - 1 = 2^1 * 1023
x0 = 3^1023 mod 2047 = 1565
The round fails; therefore, the number is surely composite.
Base 3 is a witness: it proves that 2047 is composite.



## 6. Complete probabilistic algorithm

Before using random bases, we handle simple cases:

- $n<2$: not prime;
- $n=2$ or $n=3$: prime;
- even $n>2$: composite.

For an odd $n>3$, perform $k$ independent Miller–Rabin rounds. Stop immediately if any base proves that $n$ is composite. If all rounds pass, report **probably prime**.

`SystemRandom` uses randomness supplied by the operating system. The function below is intentionally short and educational.

In [110]:
import random

secure_random = random.SystemRandom()

def is_probable_prime(n, k=10):
    """
    Return False if n is certainly composite.
    Return True if n is probably prime after k Miller–Rabin rounds.
    """
    if not isinstance(n, int):
        raise TypeError("n must be an integer")
    if k < 1:
        raise ValueError("k must be at least 1")

    if n < 2:
        return False
    if n in (2, 3):
        return True
    if n % 2 == 0:
        return False

    for _ in range(k):
        a = secure_random.randrange(2, n - 1)  # 2 <= a <= n - 2
        if not miller_rabin_round(n, a):
            return False

    return True

In [111]:
test_values = [1, 2, 3, 4, 13, 21, 97, 221, 561,1729, 2047]

for n in test_values:
    result = is_probable_prime(n, k=10)
    label = "probably prime" if result else "surely composite"
    print(f"{n:4d}: {label}")

   1: surely composite
   2: probably prime
   3: probably prime
   4: surely composite
  13: probably prime
  21: surely composite
  97: probably prime
 221: surely composite
 561: surely composite
1729: surely composite
2047: surely composite


## 7. Testing large integers and comparing with trial division

To check the result in another way, we implement **trial division**, a simple brute-force exact test.

For an odd $n$, trial division tries the possible odd divisors

$$3,5,7,\ldots,\lfloor\sqrt n\rfloor.$$

Why can it stop at $\sqrt n$? If $n=uv$ is composite and both $u$ and $v$ were greater than $\sqrt n$, then $uv>n$, which is impossible. Therefore, a composite number must have at least one factor not exceeding $\sqrt n$.

Trial division returns a definite answer, but it may require about $\sqrt n/2$ divisibility checks. Miller–Rabin uses modular exponentiation and needs only a number of operations polynomial in the bit length $\log_2 n$. This creates a dramatic difference for large inputs.

The next cell compares both methods on the same 13-digit prime. Timings depend on the computer, so the notebook measures them when it is run.

In [112]:
from math import isqrt
from time import perf_counter

def is_prime_trial_division(n):
    """Exact brute-force primality test by trial division up to sqrt(n)."""
    if n < 2:
        return False
    if n in (2, 3):
        return True
    if n % 2 == 0:
        return False

    limit = isqrt(n)
    for divisor in range(3, limit + 1, 2):
        if n % divisor == 0:
            return False
    return True

comparison_number = 1_000_000_000_039  # a 13-digit prime

start = perf_counter()
mr_result = is_probable_prime(comparison_number, k=20)
mr_time = perf_counter() - start

start = perf_counter()
trial_result = is_prime_trial_division(comparison_number)
trial_time = perf_counter() - start

print(f"Number tested: {comparison_number:,}")
print(f"Miller–Rabin:   {'probably prime' if mr_result else 'composite':16s} time = {mr_time:.6f} s")
print(f"Trial division: {'prime' if trial_result else 'composite':16s} time = {trial_time:.6f} s")
if mr_time > 0:
    print(f"Trial division took about {trial_time / mr_time:,.1f} times as long in this run.")

Number tested: 1,000,000,000,039
Miller–Rabin:   probably prime   time = 0.000936 s
Trial division: prime            time = 0.102585 s
Trial division took about 109.6 times as long in this run.


### A much larger integer

Now test

$$2^{61}-1=2{,}305{,}843{,}009{,}213{,}693{,}951.$$

Miller–Rabin handles this 19-digit integer quickly. A complete brute-force test would need to consider odd divisors up to roughly

$$\sqrt{2^{61}-1}\approx 1.52\times10^9,$$

which means about $7.6\times10^8$ odd trial divisions in the worst case. The cell therefore times Miller–Rabin and reports the brute-force workload instead of making the notebook wait for hundreds of millions of divisions.

In [113]:
large_number = 2**61 - 1

start = perf_counter()
large_result = is_probable_prime(large_number, k=20)
large_mr_time = perf_counter() - start

odd_trial_divisions = (isqrt(large_number) - 1) // 2

print(f"Large integer: {large_number:,}")
print(f"Miller–Rabin result: {'probably prime' if large_result else 'composite'}")
print(f"Miller–Rabin time:   {large_mr_time:.6f} s")
print(f"Worst-case odd trial divisions: approximately {odd_trial_divisions:,}")
print("Full brute-force timing is intentionally skipped for this input.")

Large integer: 2,305,843,009,213,693,951
Miller–Rabin result: probably prime
Miller–Rabin time:   0.001961 s
Worst-case odd trial divisions: approximately 759,250,124
Full brute-force timing is intentionally skipped for this input.


## 7. Why does the error bound contain $4^{-k}$?

This point is easier to understand one round at a time.

### Step 1: What can happen in one round?

For every odd composite number $n$, **at most one quarter** (we will see why) of the allowable bases can fool one Miller–Rabin round. In other words,

$$P(\text{a composite passes one round})\le \frac14.$$

The fraction $1/4$ is an upper bound. For many composite numbers, the actual chance is smaller. If a round fails, compositeness is certain.

### Step 2: What happens in two independent rounds?

For the composite number to survive two rounds, both randomly selected bases must fool the test. Because the bases are selected independently, the probabilities multiply:

$$\frac14\times\frac14=\frac1{16}=4^{-2}.$$

### Step 3: What happens in $k$ independent rounds?

The composite number must be lucky in every round:

$$\underbrace{\frac14\times\frac14\times\cdots\times\frac14}_{k\text{ times}}
=\left(\frac14\right)^k=\frac1{4^k}=4^{-k}.$$


A simple analogy is drawing from a bag in which at most one of four equally likely choices is misleading. After each draw, replace the choice and mix the bag, so the rounds remain independent. Getting a misleading choice once has probability at most $1/4$; doing so in every one of $k$ draws has probability at most $(1/4)^k$.

Examples:

- $k=1$: error probability at most $1/4$;
- $k=5$: at most $1/4^5=1/1024$;
- $k=10$: at most $1/4^{10}=1/1{,}048{,}576$;
- $k=20$: at most $1/4^{20}=1/1{,}099{,}511{,}627{,}776$.

This is a **one-sided error**:

- a prime always passes;
- a failed round proves that the number is composite;
- a composite can occasionally pass, so success is reported as **probably prime** rather than definitely prime.

The bound requires independently and uniformly selected random bases. Notice how the example $2047$ motivates repetition: base $2$ lies, but base $3$ reveals the compositeness.

### Why is the probability at most one quarter?

A full proof uses group theory and the Chinese Remainder Theorem. The idea, together with examples where the bound is actually attained, is sufficient here.

Write

$$n-1=2^s d,\qquad d\text{ odd}.$$

For a chosen base $a$, Miller–Rabin examines

$$a^d,\ a^{2d},\ a^{4d},\ldots,a^{2^{s-1}d}\pmod n.$$

A base passes when the chain starts with $1$, or when one of its entries is $-1$. If $n$ is composite and a base still passes, that base is called a **strong liar**.

#### The proof idea

A strong liar must satisfy two restrictive requirements.

1. **It must be coprime to $n$.** If $\gcd(a,n)>1$, powers of $a$ cannot become $1$ or $-1$ modulo $n$ in the required way. Such a base cannot be a strong liar.

2. **Its behaviour must agree modulo every prime-power factor of $n$.** If

   $$n=p_1^{e_1}p_2^{e_2}\cdots p_r^{e_r},$$

   the Chinese Remainder Theorem says that a computation modulo $n$ is equivalent to simultaneous computations modulo these prime powers. To pass, all the component squaring chains must show compatible prime-like behaviour. A mismatch exposes compositeness.

The counting argument shows that these simultaneous conditions can hold for at most one quarter of the residues coprime to $n$:

$$\#\{\text{strong liars among coprime residues}\}\leq\frac{\varphi(n)}4.$$

Thus, at least three quarters of the coprime residues are witnesses. Non-coprime bases are also witnesses, so they cannot increase the probability of error.

The fraction $1/4$ is not merely an artificial estimate. It is attained by some composite numbers when all coprime residues $1\leq a\leq n-1$ are counted.

### Numbers for which the $1/4$ bound is attained

The following examples have exactly $\varphi(n)/4$ strong liars among their coprime residues:

| Composite number $n$ | Factorization | $\varphi(n)$ | Number of strong liars | Proportion |
|---:|:---:|---:|---:|:---:|
| $15$ | $3\cdot5$ | $8$ | $2$ | $2/8=1/4$ |
| $91$ | $7\cdot13$ | $72$ | $18$ | $18/72=1/4$ |
| $703$ | $19\cdot37$ | $648$ | $162$ | $162/648=1/4$ |
| $1891$ | $31\cdot61$ | $1800$ | $450$ | $450/1800=1/4$ |

Therefore, the universal constant $1/4$ cannot be replaced by a smaller constant when the theorem is stated for all coprime residues.

### The smallest example: $n=15$

We have

$$15-1=14=2\cdot7.$$

The residues coprime to $15$ are

$$1,2,4,7,8,11,13,14.$$

Only $1$ and $14\equiv-1\pmod{15}$ pass. Hence

$$\frac{2}{8}=\frac14.$$

This attains the bound exactly.

### Why our implementation actually has a slightly smaller probability

The probabilistic implementation does not select $1$ or $n-1$. It chooses

$$a\in\{2,3,\ldots,n-2\}.$$

The two excluded residues are always strong liars:

$$1^d\equiv1\pmod n$$

and, because $d$ is odd,

$$(n-1)^d\equiv(-1)^d\equiv-1\pmod n.$$

Consequently, after removing these two automatic liars, the proportion among the bases actually selected by the program is strictly less than $1/4$ for every fixed odd composite $n$.

For example:

| $n$ | Strong liars among all coprime residues | Strong liars among program bases $2,\ldots,n-2$ | Program-base proportion |
|---:|:---:|:---:|:---:|
| $15$ | $2/8$ | $0/12$ | $0$ |
| $91$ | $18/72$ | $16/88$ | $2/11\approx0.182$ |
| $703$ | $162/648$ | $160/700$ | $8/35\approx0.229$ |
| $1891$ | $450/1800$ | $448/1888$ | $14/59\approx0.237$ |

These proportions approach $1/4$ in some examples, but remain below it because the program excludes $1$ and $n-1$.

### What “at most $1/4$” means in the algorithm

For a uniformly selected random base from $2$ to $n-2$,

$$P(\text{a composite passes one round})<\frac14.$$

For a simple universal error bound, we write

$$P(\text{a composite passes one round})\leq\frac14.$$

This is valid because a quantity strictly smaller than $1/4$ is certainly at most $1/4$.

If $k$ bases are chosen independently, the same composite number must be lucky in every round. Therefore,

$$P(\text{a composite passes all }k\text{ rounds})
\leq\underbrace{\frac14\cdot\frac14\cdots\frac14}_{k\text{ times}}
=\left(\frac14\right)^k
=4^{-k}.$$

For example:

- after $1$ round, the error probability is at most $1/4$;
- after $2$ rounds, it is at most $1/16$;
- after $3$ rounds, it is at most $1/64$;
- after $10$ rounds, it is at most $1/1{,}048{,}576$;
- after $20$ rounds, it is at most $1/1{,}099{,}511{,}627{,}776$.

### Your takeaway

- Some composites really do have strong liars occupying exactly one quarter of their coprime residues. Examples include $15$, $91$, $703$, and $1891$.
- This shows that the classical $1/4$ bound is sharp for coprime residues.
- The program selects bases only from $2$ to $n-2$, removing the two automatic liars $1$ and $n-1$.
- Therefore, its actual one-round error probability is strictly below $1/4$.
- We still use $1/4$ as a clean, safe upper bound.
- Independent repetition gives the error bound $4^{-k}$.

In [114]:
for k in (1, 5, 10, 20):
    bound = 4 ** (-k)
    print(f"k = {k:2d}: error probability <= {bound:.12g} = 1/{4**k}")

k =  1: error probability <= 0.25 = 1/4
k =  5: error probability <= 0.0009765625 = 1/1024
k = 10: error probability <= 9.53674316406e-07 = 1/1048576
k = 20: error probability <= 9.09494701773e-13 = 1/1099511627776


## 9. Correctness idea

### Why a failed round proves compositeness

Suppose the test finds a chain that neither begins with $1$ nor reaches $-1$ before ending at $1$. Such behaviour cannot occur modulo a prime, because a prime modulus has only the two square roots $\pm1$ of $1$. Therefore, the number must be composite.

### Why passing is not an absolute proof

A composite number can imitate the required behaviour for some bases. For that base, the composite number is a strong probable prime. Repeating the test with independently chosen bases makes repeated imitation very unlikely, but the result remains probabilistic.



## 10. Running-time analysis

Let $b=\lceil\log_2 n\rceil$ be the number of bits in $n$.

### Work in one round

- The exponent $d<n$ has at most $b$ bits.
- Binary modular exponentiation uses $O(\log n)=O(b)$ modular multiplications.
- The later squaring loop performs at most $s-1\le O(\log n)$ modular squarings.

Therefore, one round uses $O(\log n)$ modular multiplications.

If ordinary multiplication of two $b$-bit integers costs $O(b^2)$ bit operations, one round costs

$$O(b^3)=O((\log n)^3)$$

bit operations. With $k$ rounds, the simple bound is

$$O(kb^3)=O\!\left(k(\log n)^3\right).$$

Faster big-integer multiplication can improve this practical and theoretical cost.

### Space

Only a small number of integers of at most about $b$ bits are retained, so the extra space is $O(b)=O(\log n)$ bits, apart from implementation details.

### Early stopping

A composite candidate is rejected as soon as one witness is found. Consequently, many composite inputs finish before all $k$ rounds are performed.

## 11. Summary of the algorithm

Given $n$ and number of rounds $k$:

1. Handle $n<2$, $n\in\{2,3\}$, and even $n$.
2. Write $n-1=2^s d$ with $d$ odd.
3. Repeat $k$ times:
   1. choose random $a\in\{2,\ldots,n-2\}$;
   2. compute $x=a^d\bmod n$;
   3. pass the round if $x=1$ or $x=n-1$;
   4. otherwise repeatedly square modulo $n$, looking for $n-1$;
   5. if $n-1$ never appears, declare $n$ composite.
4. If every round passes, declare $n$ probably prime.

The crucial distinction is:

- **composite** means the algorithm has found a mathematical certificate in the form of a witness;
- **probably prime** means no witness was found among the tested bases.

## 12. One exercise

Consider $n=25$ and use base $a=2$.

1. Write $n-1$ in the form $2^s d$, where $d$ is odd.
2. Compute the complete Miller–Rabin squaring chain.
3. Decide whether base $2$ is a witness for the compositeness of $25$.
4. Verify your work by running `miller_rabin_round(25, 2, show_steps=True)`.
5. If a composite number passes $k=8$ independent rounds, what is the theoretical upper bound on this event?

Try the mathematics by hand before running the code.

In [1]:
## FUNCTIONS
def write_as_power_of_two(n):
    d = n - 1
    s = 0
    while d % 2 == 0:
        d //= 2
        s += 1
    return s, d

def miller_rabin_round(n, a, show_steps=False):
    s, d = write_as_power_of_two(n)
    x = pow(a, d, n)
    
    if show_steps:
        print(f"n - 1 = 2^{s} * {d}")
        print(f"x0 = {a}^{d} mod {n} = {x}")
    
    if x == 1 or x == n - 1:
        if show_steps:
            print("Round passes!")
        return True
    
    for i in range(1, s):
        x = (x * x) % n
        if show_steps:
            print(f"x{i} = previous squared mod {n} = {x}")
        if x == n - 1:
            if show_steps:
                print("Round passes!")
            return True
            
    if show_steps:
        print("Round fails; base is a witness that n is composite.")
    return False

# MAIN
miller_rabin_round(25, 2, show_steps=True)

n - 1 = 2^3 * 3
x0 = 2^3 mod 25 = 8
x1 = previous squared mod 25 = 14
x2 = previous squared mod 25 = 21
Round fails; base is a witness that n is composite.


False